# Pacotes

In [ ]:
import pandas as pd
import numpy as np

from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from prettytable import PrettyTable
import sklearn
from sklearn.metrics import confusion_matrix

import time
from tensorflow import keras
import tensorflow as tf
from keras.models import Sequential
from keras.models import Sequential
from keras.layers import Dense, LSTM, Bidirectional, Flatten #Global max pullin avg max pulling
from keras.layers import  Masking

from keras.callbacks import EarlyStopping, ModelCheckpoint

tf.config.list_physical_devices('GPU')

# Funcoes

In [ ]:
##############################
###### Matriz de confusão ####
##############################

def matriz_confusao(y_real,y_predito,modelo):

### Grafico ###

  tabela=confusion_matrix(y_real,y_predito)

  group_names = ["True Neg","False Pos","False Neg","True Pos"]
  group_counts = ["{0:0.0f}".format(value) for value in
                tabela.flatten()]
  group_percentages = ["{0:.5%}".format(value) for value in
                     tabela.flatten()/np.sum(tabela)]
  labels = [f"{v1}\n{v2}\n{v3}" for v1, v2, v3 in
          zip(group_names,group_counts,group_percentages)]
  labels = np.asarray(labels).reshape(2,2)
  f = plt.figure()
  f.set_figwidth(8)
  f.set_figheight(8)

  sns.heatmap(tabela, annot=labels, fmt="", cmap='Blues')

### Tabela ###
  Resultados=PrettyTable()
  Resultados.field_names=["Métrica","Resultado"]
  Resultados.title= modelo
  Resultados.align["Métrica"]="l"
  Resultados.align["Resultado"]="r"

  Resultados.add_row(["Acurácia:",round(sklearn.metrics.accuracy_score(y_real,y_predito),2)])
  Resultados.add_row(["Precisão:",round(sklearn.metrics.precision_score(y_real,y_predito),2)])
  Resultados.add_row(["Recall:",round(sklearn.metrics.recall_score(y_real,y_predito),2)])
  Resultados.add_row(["F1-Score:",round(sklearn.metrics.f1_score(y_real,y_predito),2)])

  print(Resultados)
  
  return

# Importacao dos dados

In [ ]:
df = pd.read_csv('df_comp.csv')

In [ ]:
df.replace(np.nan,-1, inplace=True)

In [ ]:
df['tempo'] =  pd.to_datetime(df['tempo'])

In [ ]:
df = df.sort_values(['n_coord','tempo'])

In [ ]:
teste = df[df.loc[:,'tempo'] > pd.to_datetime('2021-08-24 09:00:00')]

In [ ]:
treino = df[df.loc[:,'tempo'] <= pd.to_datetime('2021-08-24 09:00:00')]

In [ ]:
### Treino

X = treino[["vintequatrohoras", "Altitude_numerica", "Declividade_numerica", "graurisc", 'lon_ocr', 'lat_ocr', 'Orientacao_numerica', 'Curv_Vertical_numerica', 'Curv_Horizontal_numerica', 'Relevo_sombreado_numerico','Vegetacao_Natural_Dominante'                
,'Area_Antropica_Dominante'                   
,'legenda_2_Pecuária (pastagens)'             
,'Floresta_Ombrofila_Densa'                   
,'Formacao_Pioneira'                          
,'Floresta_Ombrofila_Densa_Submontana'        
,'Influencia_urbana'                          
,'Vegetacao_Secundaria'                       
,'Argilossolo'                                
,'Gleissolo'                                  
,'Argilossolo_Vermelho_Amarelo'               
,'Gleissolo_Melanico'                         
,'Area_Urbana'                                
,'Unidades_de_Conservacao_Protecao_Integral'  
,'Plano_de_Manejo'                            
,'flg_comunidades'                          
,'flg_agricola'                               
,'flg_exploracao_mineral'                     
,'flg_rocha'                                  
,'flg_cobertura_vegetal'                      
,'flg_afloramento_rochoso'                    
,'flg_favela'                                 
,'flg_ocupacao_desordenada' ]]

Y = treino[['ocorrencia']]

In [ ]:
### Teste

X_teste = teste[["vintequatrohoras", "Altitude_numerica", "Declividade_numerica", "graurisc", 'lon_ocr', 'lat_ocr', 'Orientacao_numerica', 'Curv_Vertical_numerica', 'Curv_Horizontal_numerica', 'Relevo_sombreado_numerico','Vegetacao_Natural_Dominante'                
,'Area_Antropica_Dominante'                   
,'legenda_2_Pecuária (pastagens)'             
,'Floresta_Ombrofila_Densa'                   
,'Formacao_Pioneira'                          
,'Floresta_Ombrofila_Densa_Submontana'        
,'Influencia_urbana'                          
,'Vegetacao_Secundaria'                       
,'Argilossolo'                                
,'Gleissolo'                                  
,'Argilossolo_Vermelho_Amarelo'               
,'Gleissolo_Melanico'                         
,'Area_Urbana'                                
,'Unidades_de_Conservacao_Protecao_Integral'  
,'Plano_de_Manejo'                            
,'flg_comunidades'                          
,'flg_agricola'                               
,'flg_exploracao_mineral'                     
,'flg_rocha'                                  
,'flg_cobertura_vegetal'                      
,'flg_afloramento_rochoso'                    
,'flg_favela'                                 
,'flg_ocupacao_desordenada' ]]

Y_teste = teste[['ocorrencia']]

In [ ]:
tempo_treino = len(treino['tempo'].unique())
n_coord_treino = len(treino['n_coord'].unique())

tempo_teste = len(teste['tempo'].unique())
n_coord_teste = len(teste['n_coord'].unique())

In [ ]:
X_train = X.to_numpy().reshape(n_coord_treino,tempo_treino,-1)


In [ ]:
Y_train = Y.to_numpy().reshape(n_coord_treino,tempo_treino,-1)

In [ ]:
class_weight_ = {0: (1/766082),
                1: (1/49)}

In [ ]:
X_train.shape

In [ ]:
Y_train.shape

In [ ]:
#Create sample_weight, an array of 1's with 0 on the missing labels
sample_weight=np.ones(len(labels))
sample_weight[ind_nan]=0
#Create dataframes with data and weights
df = pd.DataFrame({'labels': labels})
sample_weight_df = pd.DataFrame(dict(sample_weight=sample_weight))

In [ ]:
time_step = X_train.shape[1]
input_dim = X_train.shape[2] #qtd colunas 
out = Y_train.shape[2]

# LSTM
start = time.time()
model = Sequential()
model.add(Masking(mask_value=-1.,input_shape=(time_step, input_dim,))) #camada de entrada
model.add(LSTM(32,activation='elu', input_shape=(time_step, input_dim,),return_sequences=True, go_backwards=True)) # camada escondida
model.add(Dense(out, activation='sigmoid')) #camada saida

#opt = tf.keras.optimizers.Adam(learning_rate=0.01)

model.compile(loss = tf.keras.losses.BinaryCrossentropy(from_logits=False) #https://keras.io/api/losses/probabilistic_losses/ 
              , optimizer='adam', metrics = ['accuracy'
              , tf.keras.metrics.Precision()
              , tf.keras.metrics.Recall()])
              , sample_weight_mode="temporal"   #Weights)
model.summary()
hist = model.fit(X_train
                 , Y_train
                 , epochs=100
                 , validation_split=.2
                 , verbose=1, batch_size=10
                 , sample_weight=sample_weight_train #(samples, sequence_length))
end = time.time()
print("Total compile time: --------", end - start, 's')



In [ ]:
len(Y_train)

In [ ]:
# summarize history for accuracy
plt.plot(hist.history['accuracy'])
plt.plot(hist.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()

# summarize history for loss
plt.plot(hist.history['loss'])
plt.plot(hist.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()

# summarize history for accuracy
plt.plot(hist.history['precision_4'])
plt.plot(hist.history['val_precision_4'])
plt.title('model precision')
plt.ylabel('precision')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()

# summarize history for accuracy
plt.plot(hist.history['recall_4'])
plt.plot(hist.history['recall_4'])
plt.title('model recall')
plt.ylabel('recall')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()

In [ ]:
model.summary()

In [ ]:
pred_y = model.predict(X_test, verbose=1)

In [ ]:
matriz_confusao(teste[['ocorrencia']],pred_y,'1')

In [ ]:
np.mean(pred_y)

In [ ]:
max(pred_y)

In [ ]:
min(pred_y)